
# BigQuery Basics — `bq` CLI

Five questions a stakeholder actually asks about Austin's public bike-share data, and the SQL
concept each one drags in:

1. How many trips, how long do they run? → basic aggregates
2. Which stations are busiest relative to capacity? → joining a fact table to a dimension table
3. Can analysts reuse that join without copying data or re-typing it? → views, then materialized views
4. Queries are getting expensive as the table grows — how do we stop scanning everything? → partitioning
5. Narrowed to a day, still scanning every station in it — narrower still? → clustering

Each answer motivates the next question, so we go in order: join → view → materialized view →
partition → cluster.

Everything runs through `bq` (bundled with the Cloud SDK) — `gcloud` only handles auth/project
setup. No Python client here.

## The dataset
`bigquery-public-data.austin_bikeshare` — small enough to query cheaply live, shaped right for
every concept on the list:

- **`bikeshare_trips`** (~1.5M rows) — one row per trip. Has a timestamp (`start_time`, good for
  partitioning) and a station ID (good for joins/clustering). Our fact table.
- **`bikeshare_stations`** — one row per dock. Small, barely changes. Our dimension table.

Public datasets occasionally rename columns — Section 2 prints the live schema before we query
anything, so trust that over this text if they ever disagree.

Storage is free (Google hosts it); querying still counts against your bytes quota, so every
section shows the `--dry_run` estimate before running for real.

A print-friendly version of this same walkthrough exists using a small synthetic Indian employee
dataset instead, if you want a leave-behind reference.


## 1. Auth & config

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("authenticated, gcloud/bq configured for this session")
else:
    print("not in Colab — assuming ADC is set up")


In [ ]:
import time, os

PROJECT_ID = input("Enter your GCP Project ID: ").strip()
BQ_LOCATION = input("BigQuery location [US]: ").strip() or "US"
DATASET_NAME = f"bq_training_demo_{int(time.time())}"

# %%bash cells below read these as $PROJECT_ID etc, not Python interpolation
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["DATASET_NAME"] = DATASET_NAME
os.environ["BQ_LOCATION"] = BQ_LOCATION

!gcloud config set project {PROJECT_ID}
!bq mk --dataset --location={BQ_LOCATION} {PROJECT_ID}:{DATASET_NAME}
print("working dataset:", DATASET_NAME)



> 🖥️ SQL workspace's Explorer panel should show the new dataset — empty for now.



## 2. Look at the dataset first
Print the live schema before writing anything against it — cheapest way to avoid a query failing
on a renamed column.


In [ ]:
!bq show --format=prettyjson bigquery-public-data:austin_bikeshare.bikeshare_trips | head -60
!bq show --format=prettyjson bigquery-public-data:austin_bikeshare.bikeshare_stations | head -60


In [ ]:
!bq show bigquery-public-data:austin_bikeshare.bikeshare_trips
!bq show bigquery-public-data:austin_bikeshare.bikeshare_stations



> 🖥️ Same table page's Details tab shows size/row-count as cards — same numbers, nicer layout.


In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT * FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips` LIMIT 5
EOF



> 🖥️ The table's Preview tab gives you the same sample rows for free — no query, no bytes billed.


## 3. Simple queries

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT COUNT(*) AS total_trips
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
EOF


In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT
  ROUND(AVG(duration_minutes), 2) AS avg_duration_minutes,
  MAX(duration_minutes) AS max_duration_minutes,
  MIN(duration_minutes) AS min_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE duration_minutes IS NOT NULL AND duration_minutes > 0
EOF


Filtering out null/zero durations first — a handful of bad readings otherwise skews the average.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT start_station_name, COUNT(*) AS trip_count
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_name IS NOT NULL
GROUP BY start_station_name
ORDER BY trip_count DESC
LIMIT 10
EOF



> 🖥️ Run any of these in the SQL workspace — the Job information panel shows Bytes processed,
> matching what `--dry_run` would've told you upfront.


In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT
  subscriber_type,
  COUNT(*) AS trip_count,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
GROUP BY subscriber_type
ORDER BY trip_count DESC
EOF



The `OVER ()` is the interesting bit — an empty window means "aggregate across the whole result
set," which is how each row gets a percentage of the grand total without a second query.



## 4. Joins
INNER JOIN first — trips enriched with station details. Any trip whose station ID doesn't exist
in the stations table just vanishes from the result (Section 4.3 finds those on purpose).


In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT t.trip_id, t.start_time, t.start_station_name, s.status AS start_station_status,
       s.council_district, t.duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS t
INNER JOIN `bigquery-public-data.austin_bikeshare.bikeshare_stations` AS s
  ON t.start_station_id = s.station_id
LIMIT 20
EOF


In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT s.name AS station_name, s.number_of_docks, COUNT(t.trip_id) AS trip_count,
       ROUND(AVG(t.duration_minutes), 2) AS avg_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_stations` AS s
INNER JOIN `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS t
  ON s.station_id = t.start_station_id
GROUP BY s.name, s.number_of_docks
ORDER BY trip_count DESC
LIMIT 10
EOF



> 🖥️ Busiest stations relative to dock count — good for capacity planning. Try bumping `LIMIT 10`
> to `30` right in the console to see how fast you can iterate without touching this notebook.

Now the orphans — LEFT JOIN keeps every trip regardless of match, so filtering
`WHERE s.station_id IS NULL` finds exactly what the INNER JOIN above silently dropped. Common
real-world pattern worth remembering.


In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT t.start_station_id, t.start_station_name, COUNT(*) AS orphaned_trip_count
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS t
LEFT JOIN `bigquery-public-data.austin_bikeshare.bikeshare_stations` AS s
  ON t.start_station_id = s.station_id
WHERE s.station_id IS NULL
GROUP BY t.start_station_id, t.start_station_name
ORDER BY orphaned_trip_count DESC
LIMIT 20
EOF



## 5. Views
`CREATE VIEW` stores the query, not the result — every read re-runs the join live against current
data. Wrapping the station-summary join so analysts can just query the view by name.


In [ ]:
%%bash
bq query --use_legacy_sql=false --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
CREATE VIEW station_trip_summary_view AS
SELECT s.station_id, s.name AS station_name, s.number_of_docks,
       COUNT(t.trip_id) AS trip_count, ROUND(AVG(t.duration_minutes), 2) AS avg_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_stations` AS s
LEFT JOIN `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS t
  ON s.station_id = t.start_station_id
GROUP BY s.station_id, s.name, s.number_of_docks
EOF


In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT * FROM station_trip_summary_view ORDER BY trip_count DESC LIMIT 10
EOF



> 🖥️ Dataset's Explorer entry shows `station_trip_summary_view` with a "view" icon; its Details
> tab shows Table type: VIEW and the stored SQL — no data of its own.

**Views, quickly:** reusable, always live, no storage cost, and can act as an access-control layer
(an authorized view exposes a filtered slice without granting the base table). Trade-off: zero
performance benefit — every read re-executes the full join. That's what materialized views fix.



## 6. Materialized views
Same idea, but this one actually stores the result and refreshes incrementally instead of
recomputing from scratch on every read.


In [ ]:
%%bash
bq query --use_legacy_sql=false --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
CREATE MATERIALIZED VIEW daily_station_trip_counts_mv AS
SELECT DATE(start_time) AS trip_date, start_station_name, COUNT(*) AS trip_count,
       AVG(duration_minutes) AS avg_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_name IS NOT NULL
GROUP BY trip_date, start_station_name
EOF


Caveat: materialized views only support a subset of SQL — no `CURRENT_TIMESTAMP()`-style non-determinism, limited join types. Check current docs, it's expanded over time.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT trip_date, start_station_name, trip_count, ROUND(avg_duration_minutes, 2) AS avg_duration_minutes
FROM daily_station_trip_counts_mv WHERE trip_date = '2016-01-01' ORDER BY trip_count DESC LIMIT 10
EOF



> 🖥️ Dataset entry shows a distinct "materialized view" icon; Details tab has a Refresh section
> (last refresh time, staleness) — a plain view never shows this since it has nothing to refresh.

**Actually seeing the cost difference** — same logical query, dry-run both ways:


In [ ]:
%%bash
echo "against the materialized view:"
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT * FROM daily_station_trip_counts_mv WHERE trip_date = '2016-01-01' ORDER BY trip_count DESC LIMIT 10
EOF

echo ""
echo "same thing, raw aggregation against the base table:"
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT DATE(start_time) AS trip_date, start_station_name, COUNT(*) AS trip_count,
       ROUND(AVG(duration_minutes), 2) AS avg_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_name IS NOT NULL AND DATE(start_time) = '2016-01-01'
GROUP BY trip_date, start_station_name
EOF



> 🖥️ The query validator (the estimate that appears above the editor as you type, before you even
> click Run) shows this same number live — paste both queries in and watch it change.

Beyond cost: BigQuery's optimizer can transparently rewrite a query against the *base table* to
read from a matching materialized view instead — the analyst doesn't have to know it exists.
Trade-offs: restricted SQL surface, near-real-time not instant, and unlike a plain view it does
cost storage.



## 7. Partitioning
`PARTITION BY DATE(start_time)` splits the table into one physical chunk per day. A query filtered
on that date range skips every other chunk entirely — never reads it off disk. This is the single
biggest lever for cutting cost on a time-series table.


In [ ]:
%%bash
bq query --use_legacy_sql=false --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
CREATE TABLE trips_partitioned
PARTITION BY DATE(start_time) AS
SELECT * FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_time IS NOT NULL
EOF


(Filtering out NULL start times first — BigQuery dumps those into a special `__NULL__` partition that's rarely useful.)


> 🖥️ Dataset entry shows `trips_partitioned` → Details tab's Partition section confirms
> `DATE(start_time)`, and per-partition row counts show up under Table storage.

Same single-day filter, two tables — watch the bytes-processed gap:


In [ ]:
%%bash
echo "unpartitioned public table (has to check every row's date):"
bq query --use_legacy_sql=false --dry_run <<'EOF'
SELECT COUNT(*) AS trips_on_day
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE DATE(start_time) = '2016-06-15'
EOF

echo ""
echo "our partitioned copy (prunes straight to one partition):"
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT COUNT(*) AS trips_on_day
FROM trips_partitioned
WHERE DATE(start_time) = '2016-06-15'
EOF



> 🖥️ Paste both into the console and compare the query validator's live estimate — usually a
> dramatic gap (MB vs GB), and it updates instantly as you switch between them.

Three partition types exist: time-unit column (what we just did), ingestion-time (auto-partitioned
by load time, via the `_PARTITIONTIME` pseudocolumn, for data with no natural event-time column),
and integer-range (by ranges of a numeric column). Two things worth remembering in practice:
`require_partition_filter=true` forces every query to filter on the partition column — a decent
guardrail against an accidental full scan — and there's a hard cap of a few thousand partitions per
table, so partition by day or hour, not by exact timestamp.



## 8. Clustering
Sorts data *within* each partition by a column — here, `start_station_id`. Partitioning gets you to
a day; clustering narrows further within that day.


In [ ]:
%%bash
bq query --use_legacy_sql=false --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
CREATE TABLE trips_partitioned_clustered
PARTITION BY DATE(start_time)
CLUSTER BY start_station_id AS
SELECT * FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_time IS NOT NULL
EOF



> 🖥️ `trips_partitioned_clustered`'s Details tab shows both a Partition section and a Clustering
> section — the one place you'll see both optimizations on the same table at once.

Clustering's benefit doesn't always show up cleanly in `--dry_run` the way partition pruning does
— the estimator can undercount block-level pruning. Treat these as indicative:


In [ ]:
%%bash
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT COUNT(*) AS trips FROM trips_partitioned
WHERE DATE(start_time) = '2016-06-15' AND start_station_id = 1006
EOF

bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT COUNT(*) AS trips FROM trips_partitioned_clustered
WHERE DATE(start_time) = '2016-06-15' AND start_station_id = 1006
EOF



> 🖥️ For the real answer, actually run both (not just dry-run) and compare **Bytes billed** in
> Job history — that's the authoritative number, dry-run is only an estimate.


## Cleanup

In [ ]:
!bq rm -f -t {PROJECT_ID}:{DATASET_NAME}.daily_station_trip_counts_mv
!bq rm -f -t {PROJECT_ID}:{DATASET_NAME}.station_trip_summary_view
!bq rm -f -t {PROJECT_ID}:{DATASET_NAME}.trips_partitioned
!bq rm -f -t {PROJECT_ID}:{DATASET_NAME}.trips_partitioned_clustered
!bq rm -f -r -d {PROJECT_ID}:{DATASET_NAME}
print("dataset and everything in it: gone")


In [ ]:
!bq show {PROJECT_ID}:{DATASET_NAME} 2>&1 | head -3
